# Part 3 · Challenge: you are the station manager

```
            6  City Exit (Y)
            |
            4  Mall Junction
            |  \
   (wide)   |    \   (narrow diagonal)
            |      \
   0 ───────2 ───────3 ─────────── 5  Platform (X)
  West     Plaza  ↑  Hall
  Gate (A)    NARROW  |
                      1  South Gate (B)
```

Through signage, staff and app messages you can steer **which destination people from each gate
head to** (think *"Platform entrance via the South Gate"*). In model terms: **you choose the OD weights.**

### Rules
* Gate demand is fixed (you cannot change it).
* The train company requires that **at least 60%** of all travellers are directed to the **Platform X**.
  Both gates have the same demand, so the average of your two "share to X" values must be ≥ 0.60.
* **Score = time spent inside (person-hours) + 0.25 h for every person turned away.** Lower is better.
* Route-choice parameters and corridor widths stay at their defaults (until the bonus round).

Enter your best score on the shared leaderboard together with your two shares.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# On Google Colab this installs PedNStream and fetches the workshop files.
# On your own laptop, follow README.md instead and this cell does nothing.
import sys, os
if "google.colab" in sys.modules and not os.path.exists("workshop_tools.py"):
    WORKSHOP_REPO = "https://github.com/WaimenMak/pednstream-workshop.git"  # <- facilitator: set this
    !git clone -q {WORKSHOP_REPO} ws && cp -r ws/* .
    %pip install -q -r requirements.txt

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 90
from workshop_tools import run, compare, compare_maps, switch, ramp
print("Ready ✔")

def score(share_A_to_X, share_B_to_X, **extra):
    """Run your strategy and return its score (lower is better)."""
    assert 0 <= share_A_to_X <= 1 and 0 <= share_B_to_X <= 1, "shares must be between 0 and 1"
    avg = (share_A_to_X + share_B_to_X) / 2
    res = run(od={"0_5": share_A_to_X, "0_6": 1 - share_A_to_X,
                  "1_5": share_B_to_X, "1_6": 1 - share_B_to_X}, **extra)
    k = res.kpis()
    s = k["time spent inside (person-hours)"] + 0.25 * k["turned away at gates"]
    ok = avg >= 0.6 - 1e-9
    print(f"Share to X: A={share_A_to_X:.2f}  B={share_B_to_X:.2f}  ->  overall {avg:.0%}"
          f"  {'✔ meets 60% rule' if ok else '✘ BREAKS the 60% rule (score not valid)'}")
    print(f"SCORE = {s:.1f}   (time inside {k['time spent inside (person-hours)']:.1f} h,"
          f" turned away {k['turned away at gates']})")
    return res

### Your attempts
Start from the naive strategy: both gates send 60% to the Platform.

In [ ]:
r = score(0.6, 0.6)
r.dashboard();

In [ ]:
# attempt 2

In [ ]:
# attempt 3

### Systematic search (optional)
Instead of guessing, scan the possibilities. Fill in the loop below and make a heat-map of the
score for every combination of shares.

In [ ]:
import numpy as np
shares = np.linspace(0, 1, 6)
# results = {}
# for a in shares:
#     for b in shares:
#         ...

### Bonus round: one infrastructure upgrade
You get budget to widen **one** corridor by 2 m, e.g. `score(a, b, links={"2_3": {"width": 4}})`.
Default widths: 2_3 = 2 m, 3_4 = 2 m, 2_4 = 4 m, all others 3 m.
**Does your best OD strategy change once the network changes?**

In [ ]:
# bonus attempts

### Reflection
1. Describe your winning strategy in words a station manager would understand.
2. Why does it work? Refer to specific corridors and to *route overlap*.
3. Name one thing the model leaves out that could make your strategy fail in real life.